# 08 — Live Data Pipeline

Every prior notebook in this project reads `data/raw/telco_churn.csv` fresh, in full,
on every run. That's fine for a one-off analysis, but it isn't how a real subscription
business's data actually arrives — customers sign up continuously, and billing events
happen every month, not all at once in a single file drop.

This notebook demonstrates the persistent, incremental alternative built in
`src/database.py`: a DuckDB-backed store with an `ingest_customers()` upsert path and
an `ingest_month()` append-only path, plus an `ingestion_log` table that records every
load — what happened, how many rows, when. This is also the foundation the drift
detection notebook (10) builds on: monitoring for drift only makes sense against a
system that receives new data incrementally over time, not one that reloads a static
file.

**What this is not:** a claim that this connects to any real production database.
It's a demonstration of the *pattern* — persistent storage, upsert-safe customer
records, append-only billing history, and a full ingestion audit trail — using the
same synthetic dataset as the rest of the project, simulated as if it arrived over
time rather than all at once.


In [1]:
import sys
sys.path.insert(0, '../')

import pandas as pd
from src.preprocessing import load_raw
from src import database as db

con = db.get_connection()
db.init_schema(con)
print('Schema initialized at', db.DB_PATH)


Schema initialized at data/vertex_churn.db


## Ingesting the customer base

`ingest_customers()` upserts on `customerID` — running this twice with the same data
updates existing rows instead of duplicating them, which matters because a customer's
`Churn` flag and `tenure` genuinely do change over their lifecycle in a real system.


In [2]:
customers = load_raw()
n = db.ingest_customers(con, customers)
print(f'Ingested {n:,} customer records')

# running it again should update, not duplicate
n_again = db.ingest_customers(con, customers)
total_rows = db.query(con, 'SELECT COUNT(*) as n FROM customers').iloc[0]['n']
print(f'Re-ingested {n_again:,} records; total rows in table: {total_rows:,} (should equal original count, not double it)')


Ingested 7,043 customer records
Re-ingested 7,043 records; total rows in table: 7,043 (should equal original count, not double it)


## Simulating monthly billing arrival

Instead of loading all 71 months of the revenue panel in one call, we ingest one
month at a time — this is the loop a real scheduled job would run, and it's what lets
notebook 10 (drift detection) check each newly-arrived month against a fixed training
baseline as it comes in, rather than checking a dataset that was already complete
from the start.


In [3]:
panel = pd.read_csv('../data/raw/monthly_revenue_panel.csv')
max_month = panel['month'].max()

for m in range(1, max_month + 1):
    db.ingest_month(con, panel, m)

print(f'Ingested {max_month} months of billing events')
print()
print(db.query(con, 'SELECT COUNT(*) as total_billing_events FROM monthly_revenue'))


Ingested 71 months of billing events

   total_billing_events
0                202085


## The ingestion audit trail

Every ingestion call is logged — this is the same audit-trail discipline used
elsewhere in the project (e.g. `src/drift.py`'s retrain-reason logging): if a
downstream number looks wrong, the ingestion log is the first place to check *what
data arrived, and when*, rather than re-deriving it from scratch.


In [4]:
history = db.ingestion_history(con)
print(f'{len(history)} ingestion events logged')
history.tail(15)


73 ingestion events logged


,ingestion_id,table_name,rows_ingested,ingested_at,detail
58,59,monthly_revenue,626,2026-08-21 14:53:18.944987,month 57: 626 billing events
59,60,monthly_revenue,572,2026-08-21 14:53:18.952477,month 58: 572 billing events
60,61,monthly_revenue,511,2026-08-21 14:53:18.960192,month 59: 511 billing events
61,62,monthly_revenue,443,2026-08-21 14:53:18.967622,month 60: 443 billing events
62,63,monthly_revenue,407,2026-08-21 14:53:18.974738,month 61: 407 billing events
63,64,monthly_revenue,372,2026-08-21 14:53:18.981849,month 62: 372 billing events
64,65,monthly_revenue,332,2026-08-21 14:53:18.989479,month 63: 332 billing events
65,66,monthly_revenue,299,2026-08-21 14:53:18.997058,month 64: 299 billing events
66,67,monthly_revenue,255,2026-08-21 14:53:19.004489,month 65: 255 billing events
67,68,monthly_revenue,219,2026-08-21 14:53:19.011222,month 66: 219 billing events


## Querying the live store

Once data lives in a persistent database rather than an in-memory DataFrame, every
other notebook and the dashboard can query it directly with SQL — consistent with the
project's existing `sql/` query files, now running against real persisted tables
instead of a CSV loaded into DuckDB fresh each time.


In [5]:
db.query(con, '''
    SELECT Contract, COUNT(*) as customers, ROUND(AVG(MonthlyCharges), 2) as avg_mrr
    FROM customers
    GROUP BY Contract
    ORDER BY customers DESC
''')


,Contract,customers,avg_mrr
0,Month-to-month,3857,67.30
1,Two year,1707,66.45
2,One year,1479,66.12
